In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..") / "src"))


### Boiling and metling points, heat capacity and density

#### Training

In [ ]:
from filters.thermo import ThermoModelConfig, train_property_ensemble

cfg = ThermoModelConfig(
    prop_name="density",
    smiles_col="SMILES",
    target_col="RHO",
    ensemble_size=5,
    epochs=50,
)

ckpts = train_property_ensemble(
    csv_path="../data/thermo/RHO/RHO.csv",
    out_dir="checkpoints/thermo",
    config=cfg,
)
ckpts

#### Inference

In [ ]:
import pandas as pd
from filters.thermo import predict_property_ensemble, apply_uncertainty_filter
from pathlib import Path

df = pd.read_csv("/home/rajabiko/projects/BO_SDL/Repo/data/thermo/BP/BP.csv")
property = 'BP'
ckpt_paths = sorted(Path(f"checkpoints/thermo/{property}").glob("model_*.ckpt"))

df_pred = predict_property_ensemble(
    df,
    checkpoint_paths=ckpt_paths,
    smiles_col="SMILES",
    pred_col=f"{property}_pred",
    std_col=f"{property}_std",
    output_dir=Path(f"checkpoints/thermo/{property}")
)   

"""
certain, uncertain = apply_uncertainty_filter(
    df_pred,
    std_col=f"{property}_std",
    threshold=0.05,   # you choose based on calibration
    status_col=f"{property}_status"
)
"""

### Viscsoity

In [ ]:
import pandas as pd
from filters.viscosity import (
    add_viscosity_predictions_from_csv,
    ViscosityPredConfig
)

cfg = ViscosityPredConfig(
    checkpoint_dir="checkpoints/excess_props/nist_dippr_model/nist_dippr_model",
    smi2="c1ccccc1",
    molfrac1=1,
    T=298,
    n_models=25,
    threshold=0.022,
    num_workers=0,
)

df_out = add_viscosity_predictions_from_csv(
    "/home/rajabiko/projects/BO_SDL/Repo/data/thermo/MIU/MIU.csv",
    config=cfg,
    output_csv="/home/rajabiko/projects/BO_SDL/Repo/data/thermo/MIU/MIU_with_preds.csv",
)

df_out.head()